In [23]:
import pandas as pd
import os
import logging

Run_name = 'Run_285'

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tropomi_merging.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def merge_and_save_dataframes(Run_name):
    # Path to the original file with t2m data
    original_file_path = '/net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv'
    
    # Path to the new processed data
    new_file_path = f'../data/{Run_name}/valid_tropomi_emissions_with_qa.csv'
    
    # Output directory and filename
    output_dir = os.path.dirname(new_file_path)
    output_path = os.path.join(output_dir, 'updated_tropomi_emissions.csv')
    
    logger.info(f"Loading original data from: {original_file_path}")
    original_df = pd.read_csv(original_file_path)
    logger.info(f"Loaded {len(original_df)} rows from original data")
    
    logger.info(f"Loading new processed data from: {new_file_path}")
    new_df = pd.read_csv(new_file_path)
    logger.info(f"Loaded {len(new_df)} rows from new processed data")
    
    # Create a merged dataframe based on identifying columns
    # Assuming location, latitude, longitude, and utc_time can uniquely identify a row
    logger.info("Creating mapping between original and new data")
    
    # Create matching keys in both dataframes
    original_df['match_key'] = original_df['location'].astype(str) + '_' + \
                               original_df['latitude'].astype(str) + '_' + \
                               original_df['longitude'].astype(str) + '_' + \
                               original_df['utc_time'].astype(str)
    
    new_df['match_key'] = new_df['location'].astype(str) + '_' + \
                          new_df['latitude'].astype(str) + '_' + \
                          new_df['longitude'].astype(str) + '_' + \
                          new_df['utc_time'].astype(str)
    
    # Create a dictionary mapping from match_key to new values
    new_values_dict = {}
    for idx, row in new_df.iterrows():
        new_values_dict[row['match_key']] = {
            'plume_label': row['plume_label'],
            'no2_std_50km': row['no2_std_50km'], 
            # Add any other columns you want to update here
        }
    
    # Count matches for reporting
    match_count = 0
    
    # Update the original dataframe with new values
    logger.info("Updating original dataframe with new values")
    for idx, row in original_df.iterrows():
        match_key = row['match_key']
        if match_key in new_values_dict:
            match_count += 1
            original_df.at[idx, 'plume_label'] = new_values_dict[match_key]['plume_label']
            original_df.at[idx, 'no2_std_50km'] = new_values_dict[match_key]['no2_std_50km']
            # Update any other columns here
    
    logger.info(f"Updated {match_count} rows out of {len(original_df)} original rows")
    
    # Drop the no2_var_50km column and the temporary match_key column
    logger.info("Dropping unwanted columns")
    columns_to_drop = ['no2_var_50km', 'match_key']
    original_df = original_df.drop(columns=columns_to_drop, errors='ignore')
    
    # Save the updated dataframe
    logger.info(f"Saving updated dataframe to: {output_path}")
    original_df.to_csv(output_path, index=False)
    logger.info(f"Successfully saved {len(original_df)} rows to {output_path}")
    
    return output_path

if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("Starting TROPOMI data merge")
    logger.info("=" * 80)
    
    output_file = merge_and_save_dataframes(Run_name)
    
    logger.info("=" * 80)
    logger.info(f"Merge completed. Output saved to: {output_file}")
    logger.info("=" * 80)

2025-05-24 19:14:52,587 - INFO - ================================================================================
2025-05-24 19:14:52,588 - INFO - Starting TROPOMI data merge
2025-05-24 19:14:52,596 - INFO - ================================================================================
2025-05-24 19:14:52,600 - INFO - Loading original data from: /net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv
2025-05-24 19:14:59,759 - INFO - Loaded 639408 rows from original data
2025-05-24 19:14:59,760 - INFO - Loading new processed data from: ../data/Run_285/valid_tropomi_emissions_with_qa.csv
2025-05-24 19:15:01,803 - INFO - Loaded 639901 rows from new processed data
2025-05-24 19:15:01,804 - INFO - Creating mapping between original and new data
2025-05-24 19:15:34,077 - INFO - Updating original dataframe with new values
2025-05-24 19:16:26,199 - INFO - Updated 639408 rows out of 639408 original rows
2025-05-24 19:16:26,200 - INFO - Dropping unwanted colu

In [24]:
# load the full power_plants DF
import pandas as pd
import os
import logging

power_plants = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_combined_nearby_stats_parallel_debug.csv')

# filter: no other plants within 20 km, but at least one city within 20 km
mask = (
    (power_plants['nearby_plants_count_20km'] == 0) &
    (power_plants['nearby_cities_count_20km'] == 0)
)

# apply filter and take the first 100 (or, if you have a ranking metric, sort before .head)
top100 = power_plants.loc[mask].head(100)
top50 = power_plants.loc[mask].head(50)
top20 = power_plants.loc[mask].head(20)

tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_emissions.csv',
)

filtered_100 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top100['Facility ID'])]
filtered_50 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top50['Facility ID'])]
filtered_20 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top20['Facility ID'])]

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = tropomi_combined_dropna.iloc[:,10:]
# X = df.loc['surface_albedo']

y = tropomi_combined_dropna["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.735248119360035
Precision: 0.5676718985754315
Recall: 0.7106760867432048
F1 Score: 0.6311753107400024
Confusion Matrix:
 [[65055 22063]
 [11794 28970]]
AUC: 0.8061814238312022
Classification Report:
               precision    recall  f1-score   support

       False       0.85      0.75      0.79     87118
        True       0.57      0.71      0.63     40764

    accuracy                           0.74    127882
   macro avg       0.71      0.73      0.71    127882
weighted avg       0.76      0.74      0.74    127882

Kappa: 0.42869124020096705


In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_100.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_100["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.7467501203659124
Precision: 0.6698484981171732
Recall: 0.7137258561164505
F1 Score: 0.6910914347668956
Confusion Matrix:
 [[12514  3770]
 [ 3068  7649]]
AUC: 0.8177356586081792
Classification Report:
               precision    recall  f1-score   support

       False       0.80      0.77      0.79     16284
        True       0.67      0.71      0.69     10717

    accuracy                           0.75     27001
   macro avg       0.74      0.74      0.74     27001
weighted avg       0.75      0.75      0.75     27001

Kappa: 0.4768708295412506


In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_50.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_50["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.7426497815300304
Precision: 0.7942356262071014
Recall: 0.7189349112426036
F1 Score: 0.754711653843439
Confusion Matrix:
 [[4682 1385]
 [2090 5346]]
AUC: 0.8264438709469203
Classification Report:
               precision    recall  f1-score   support

       False       0.69      0.77      0.73      6067
        True       0.79      0.72      0.75      7436

    accuracy                           0.74     13503
   macro avg       0.74      0.75      0.74     13503
weighted avg       0.75      0.74      0.74     13503

Kappa: 0.4854579603085828


In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_20.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_20["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.7554131300883423
Precision: 0.8815461346633416
Recall: 0.7326424870466322
F1 Score: 0.8002263723825693
Confusion Matrix:
 [[1533  380]
 [1032 2828]]
AUC: 0.846860992012654
Classification Report:
               precision    recall  f1-score   support

       False       0.60      0.80      0.68      1913
        True       0.88      0.73      0.80      3860

    accuracy                           0.76      5773
   macro avg       0.74      0.77      0.74      5773
weighted avg       0.79      0.76      0.76      5773

Kappa: 0.4917336563035688


In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt
import pandas as pd
import os
import logging

Run_name = 'Run_285'

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tropomi_merging.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def merge_and_save_dataframes(Run_name):
    # Path to the original file with t2m data
    original_file_path = '/net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv'
    
    # Path to the new processed data
    new_file_path = f'../data/{Run_name}/valid_tropomi_emissions_with_qa.csv'
    
    # Output directory and filename
    output_dir = os.path.dirname(new_file_path)
    output_path = os.path.join(output_dir, 'updated_tropomi_emissions.csv')
    
    logger.info(f"Loading original data from: {original_file_path}")
    original_df = pd.read_csv(original_file_path)
    logger.info(f"Loaded {len(original_df)} rows from original data")
    
    logger.info(f"Loading new processed data from: {new_file_path}")
    new_df = pd.read_csv(new_file_path)
    logger.info(f"Loaded {len(new_df)} rows from new processed data")
    
    # Create a merged dataframe based on identifying columns
    # Assuming location, latitude, longitude, and utc_time can uniquely identify a row
    logger.info("Creating mapping between original and new data")
    
    # Create matching keys in both dataframes
    original_df['match_key'] = original_df['location'].astype(str) + '_' + \
                               original_df['latitude'].astype(str) + '_' + \
                               original_df['longitude'].astype(str) + '_' + \
                               original_df['utc_time'].astype(str)
    
    new_df['match_key'] = new_df['location'].astype(str) + '_' + \
                          new_df['latitude'].astype(str) + '_' + \
                          new_df['longitude'].astype(str) + '_' + \
                          new_df['utc_time'].astype(str)
    
    # Create a dictionary mapping from match_key to new values
    new_values_dict = {}
    for idx, row in new_df.iterrows():
        new_values_dict[row['match_key']] = {
            'plume_label': row['plume_label'],
            'no2_std_50km': row['no2_std_50km'], 
            # Add any other columns you want to update here
        }
    
    # Count matches for reporting
    match_count = 0
    
    # Update the original dataframe with new values
    logger.info("Updating original dataframe with new values")
    for idx, row in original_df.iterrows():
        match_key = row['match_key']
        if match_key in new_values_dict:
            match_count += 1
            original_df.at[idx, 'plume_label'] = new_values_dict[match_key]['plume_label']
            original_df.at[idx, 'no2_std_50km'] = new_values_dict[match_key]['no2_std_50km']
            # Update any other columns here
    
    logger.info(f"Updated {match_count} rows out of {len(original_df)} original rows")
    
    # Drop the no2_var_50km column and the temporary match_key column
    logger.info("Dropping unwanted columns")
    columns_to_drop = ['no2_var_50km', 'match_key']
    original_df = original_df.drop(columns=columns_to_drop, errors='ignore')
    
    # Save the updated dataframe
    logger.info(f"Saving updated dataframe to: {output_path}")
    original_df.to_csv(output_path, index=False)
    logger.info(f"Successfully saved {len(original_df)} rows to {output_path}")
    
    return output_path

if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("Starting TROPOMI data merge")
    logger.info("=" * 80)
    
    output_file = merge_and_save_dataframes(Run_name)
    
    logger.info("=" * 80)
    logger.info(f"Merge completed. Output saved to: {output_file}")
    logger.info("=" * 80)
    

# load the full power_plants DF
power_plants = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_combined_nearby_stats_parallel_debug.csv')

# filter: no other plants within 20 km, but at least one city within 20 km
mask = (
    (power_plants['nearby_plants_count_20km'] == 0) &
    (power_plants['nearby_cities_count_20km'] == 0)
)

# apply filter and take the first 100 (or, if you have a ranking metric, sort before .head)
top100 = power_plants.loc[mask].head(100)
top50 = power_plants.loc[mask].head(50)
top20 = power_plants.loc[mask].head(20)

tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_emissions.csv',
)

filtered_100 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top100['Facility ID'])]
filtered_50 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top50['Facility ID'])]
filtered_20 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top20['Facility ID'])]

X = tropomi_combined_dropna.iloc[:,10:]
# X = df.loc['surface_albedo']

y = tropomi_combined_dropna["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))
# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_100.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_100["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)
# print(X)
X = filtered_50.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_50["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))

# print(X)
X = filtered_20.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_20["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


2025-05-24 19:18:25,544 - INFO - ================================================================================
2025-05-24 19:18:25,545 - INFO - Starting TROPOMI data merge
2025-05-24 19:18:25,548 - INFO - ================================================================================


2025-05-24 19:18:25,552 - INFO - Loading original data from: /net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv
2025-05-24 19:18:32,990 - INFO - Loaded 639408 rows from original data
2025-05-24 19:18:32,991 - INFO - Loading new processed data from: ../data/Run_285/valid_tropomi_emissions_with_qa.csv
2025-05-24 19:18:35,152 - INFO - Loaded 639901 rows from new processed data
2025-05-24 19:18:35,153 - INFO - Creating mapping between original and new data


KeyboardInterrupt: 